# Bulk answer generation

Generate answers for a configurable number of rows from a question JSON file. The default is `Q_S1.json`; change `QUESTION_PATH` for another compatible file.

Reference answers and source-of-truth fields are saved for later evaluation, but only the question text enters retrieval and generation.

## Safety, parallelism, and resume

- `LIVE = False` creates complete dry-run records without OpenRouter calls.
- `LIVE = True` sends paid requests and requires `OPENROUTER_API_KEY`.
- `MAX_WORKERS` controls bounded parallelism. Begin conservatively because rate limits and spend apply.
- There are no automatic API retries.
- The main notebook thread checkpoints each completed worker result.
- To resume, set `RESUME_RUN_DIR` to the exact existing run directory.

In [1]:
import csv
import json
import os
import statistics
import sys
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from mobile_rag.answer_generation import GenerationConfig, PROMPT_PATH, PROMPT_VERSION, PROMPT_SHA256, generation_identity
from mobile_rag.bulk_answer_generation import (
    benchmark_identity,
    RUNTIME_SHA256,
    validate_resume,
    validate_checkpoint,
    load_contexts,
    load_record_context,
    process_question,
    select_questions,
    write_record_context,
    write_results,
)
from mobile_rag.context_preparation import ContextBudget
from mobile_rag.environment import openrouter_api_key
from mobile_rag.retrieval import digest, new_run_dir, write_json

## Configuration and saved files

Set `NUMBER_OF_QUESTIONS` to a positive integer, such as `5`, to run the first five questions. Set it to `None` to run every question in the selected file. If the number exceeds the available rows, all rows run.

This notebook creates one timestamped directory under `OUTPUT_ROOT` and saves:

- `run_manifest.json`: benchmark, index, prompt, model, context, concurrency, and question-selection configuration.
- `records.jsonl`: one compact checkpoint per question, with the original question row, retrieval hits, neighbor expansion, context indexes (labels, chunk IDs, passage IDs), citations, generated result, usage, latency, and statuses. It points at `contexts.json` instead of embedding passage text.
- `contexts.json`: all prepared context packages in one file, keyed by record key, including `context_text` and source passages.
- `results.json`: one review object per question with the question, generated answer, ground-truth answer, context ID, citations, and related fields.
- `results.csv`: compact review table with the same core fields plus usage and timing.
- `summary.json`: completion counts, status counts, timing, and missing keys.

Saving is controlled here in the notebook. Worker functions return data and do not save files.

In [2]:
from mobile_rag.retrieval_hybrid import HybridRetriever, RetrievalConfig, latest_index

QUESTION_PATH = ROOT / "data/questions/Q_S1.json"  # Change this path when needed.
ENABLE_BM25 = True
ENABLE_EMBEDDINGS = True
RETRIEVAL_CONFIG = RetrievalConfig(ENABLE_BM25, ENABLE_EMBEDDINGS)
RETRIEVAL_CONFIG.validate()
QUERY_SOURCE = "question"  # Set to "topic" for an explicitly identified benchmark ablation.
INDEX_DIR = latest_index(ROOT, RETRIEVAL_CONFIG)
OUTPUT_ROOT = ROOT / "artifacts/05_2_bulk_answer_generation"
RESUME_RUN_DIR = None  # Example: OUTPUT_ROOT / "20260913_120000"

NUMBER_OF_QUESTIONS = 5  # Positive integer = first N rows; None = all rows.
LIVE = True
MAX_WORKERS = 4
GENERATION_CONFIG = GenerationConfig(max_output_tokens=1024, timeout_seconds=60, provider="DeepInfra")
CONTEXT_BUDGET = ContextBudget(total_chars=20000, instruction_reserve=7000, answer_reserve=4000)

if type(MAX_WORKERS) is not int or not 1 <= MAX_WORKERS <= 16:
    raise ValueError("MAX_WORKERS must be between 1 and 16")
print({
    "question_path": str(QUESTION_PATH.resolve()),
    "index": str(INDEX_DIR.resolve()),
    "output_root": str(OUTPUT_ROOT.resolve()),
    "number_of_questions": NUMBER_OF_QUESTIONS,
    "live": LIVE,
    "workers": MAX_WORKERS,
    "api_key_available": bool(openrouter_api_key()),
})

{'question_path': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\data\\questions\\Q_S1.json', 'index': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\03_retrieval_enhanced\\20260913_154703', 'output_root': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\05_2_bulk_answer_generation', 'number_of_questions': 5, 'live': True, 'workers': 4, 'api_key_available': True}


## Load and freeze inputs

The benchmark file is hashed before processing. Composite record keys use the filename and row ID, for example `Q_S1:1`. A live run stops before dispatch when the API key is unavailable.

In [3]:
benchmark = benchmark_identity(QUESTION_PATH)
selected_questions = select_questions(benchmark["questions"], NUMBER_OF_QUESTIONS)
if LIVE and not openrouter_api_key():
    raise RuntimeError("LIVE=True requires OPENROUTER_API_KEY; no requests were started.")

with HybridRetriever(INDEX_DIR, RETRIEVAL_CONFIG):
    pass  # Fail before scheduling requests if the model/index is missing or incompatible.

run_config = {
    "retrieval_config": asdict(RETRIEVAL_CONFIG),
    "dense_manifest_sha256": digest(INDEX_DIR / "dense_manifest.json") if ENABLE_EMBEDDINGS else None,
    "run_schema": "bulk-answer-run/v1",
    "dataset": benchmark["dataset"],
    "question_path": benchmark["path"],
    "benchmark_sha256": benchmark["sha256"],
    "available_question_count": benchmark["question_count"],
    "selected_question_count": len(selected_questions),
    "number_of_questions": NUMBER_OF_QUESTIONS,
    "index_dir": str(INDEX_DIR.resolve()),
    "retrieval_database_sha256": digest(INDEX_DIR / "retrieval.sqlite"),
    "passage_database_sha256": digest(INDEX_DIR / "passage.sqlite"),
    "enhancement_manifest_sha256": digest(INDEX_DIR / "enhancement_manifest.json"),
    "prompt_version": PROMPT_VERSION,
    "prompt_sha256": PROMPT_SHA256,
    "generation_identity": generation_identity(GENERATION_CONFIG),
    "runtime_sha256": RUNTIME_SHA256,
    "query_source": QUERY_SOURCE,
    "generation_config": asdict(GENERATION_CONFIG),
    "context_budget": asdict(CONTEXT_BUDGET),
    "live": LIVE,
    "max_workers": MAX_WORKERS,
}
print({
    "dataset": benchmark["dataset"],
    "available": benchmark["question_count"],
    "selected": len(selected_questions),
    "benchmark_sha256": benchmark["sha256"],
})

c:\Users\PK\Desktop\projects\mobile_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dataset': 'Q_S1', 'available': 100, 'selected': 5, 'benchmark_sha256': '8ca342ad54ecee63d357b6fb0034672dbd01fce73c0fbf84d2b8998e37207b93'}


## Create or resume the run

A new run writes its manifest before submitting work. Resume compares all result-affecting settings except `MAX_WORKERS`, which may be changed safely because it affects scheduling rather than individual request content.

In [4]:
if RESUME_RUN_DIR is None:
    RUN_DIR = new_run_dir(OUTPUT_ROOT)
    write_json(RUN_DIR / "run_manifest.json", {**run_config, "created_at_utc": datetime.now(UTC).isoformat()})
else:
    RUN_DIR = Path(RESUME_RUN_DIR).resolve()
    saved_config = json.loads((RUN_DIR / "run_manifest.json").read_text(encoding="utf-8"))
    validate_resume(saved_config, run_config)

RECORDS_PATH = RUN_DIR / "records.jsonl"
CONTEXTS_PATH = RUN_DIR / "contexts.json"
print("Save folder:", RUN_DIR.resolve())

Save folder: C:\Users\PK\Desktop\projects\mobile_rag\artifacts\05_2_bulk_answer_generation\20260913_160742


## Load checkpoints

JSONL is written with ASCII escaping so embedded Unicode line separators cannot split records. Every existing record must match the benchmark identity. Worker errors are retained as outcomes; they are not silently retried.

In [5]:
records_by_key = {}
if RECORDS_PATH.exists():
    with RECORDS_PATH.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            record = json.loads(line)
            if record.get("dataset") != benchmark["dataset"] or record.get("benchmark_sha256") != benchmark["sha256"]:
                raise RuntimeError(f"Checkpoint identity mismatch on line {line_number}")
            validate_checkpoint(record, run_config)
            key = record["record_key"]
            if key in records_by_key:
                raise RuntimeError(f"Duplicate checkpoint record: {key}")
            records_by_key[key] = record

contexts_by_key = load_contexts(RUN_DIR)
pending = [
    row for row in selected_questions
    if f"{benchmark['dataset']}:{row['id']}" not in records_by_key
]
print({"already_completed": len(records_by_key), "pending": len(pending), "saved_contexts": len(contexts_by_key)})

{'already_completed': 0, 'pending': 5, 'saved_contexts': 0}


## Process pending questions in parallel

Each worker opens independent read-only SQLite connections and executes retrieval, context preparation, and generation. The main thread alone writes checkpoint records, flushing each result to disk before moving on.

In [6]:
batch_started = time.perf_counter()
if pending:
    with (
        RECORDS_PATH.open("a", encoding="utf-8", newline="\n") as checkpoint,
        ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool,
    ):
        futures = {
            pool.submit(
                process_question,
                row,
                dataset=benchmark["dataset"],
                benchmark_sha256=benchmark["sha256"],
                index_dir=INDEX_DIR,
                live=LIVE,
                generation_config=GENERATION_CONFIG,
                context_budget=CONTEXT_BUDGET,
                retrieval_config=RETRIEVAL_CONFIG,
                query_source=QUERY_SOURCE,
            ): row
            for row in pending
        }
        for future in as_completed(futures):
            record = future.result()
            compact = write_record_context(RUN_DIR, record, contexts_by_key)
            checkpoint.write(json.dumps(compact, ensure_ascii=True, sort_keys=True) + "\n")
            checkpoint.flush()
            os.fsync(checkpoint.fileno())
            records_by_key[compact["record_key"]] = compact
            print(f"[{len(records_by_key)}/{len(selected_questions)}] {compact['record_key']}: {compact['pipeline_status']}")

batch_seconds = time.perf_counter() - batch_started
print({"newly_processed": len(pending), "batch_seconds": batch_seconds, "save_folder": str(RUN_DIR.resolve())})

[1/5] Q_S1:4: api_error
[2/5] Q_S1:2: api_error
[3/5] Q_S1:1: api_error
[4/5] Q_S1:3: api_error
[5/5] Q_S1:5: api_error
{'newly_processed': 5, 'batch_seconds': 1.6848123000236228, 'save_folder': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\05_2_bulk_answer_generation\\20260913_160742'}


## Save the compact review table and summary

Full evidence text remains in `contexts.json`. `results.json` keeps the question, generated answer, ground-truth answer, and context ID for review. `records.jsonl` keeps checkpoint indexes and a pointer into `contexts.json`. The CSV adds usage and timing. API errors, invalid responses, abstentions, and missing records remain visible rather than being removed from totals.

In [7]:
ordered_keys = [f"{benchmark['dataset']}:{row['id']}" for row in selected_questions]
ordered_records = [records_by_key[key] for key in ordered_keys if key in records_by_key]
overview = []
for record in ordered_records:
    row = record["question_record"]
    generated = record.get("generation") or {}
    answer = generated.get("answer") or {}
    context = record.get("context") or {}
    retrieval = record.get("retrieval") or {}
    usage = generated.get("usage") or {}
    overview.append({
        "record_key": record["record_key"],
        "question_id": row.get("id"),
        "category": row.get("category"),
        "topic": row.get("topic"),
        "question": row.get("question"),
        "reference_answer": row.get("answer"),
        "source_of_truth": row.get("source_of_truth"),
        "pipeline_status": record["pipeline_status"],
        "answer_status": answer.get("status"),
        "generated_answer": answer.get("answer"),
        "generated_reason": answer.get("reason"),
        "citations": json.dumps(answer.get("citations"), ensure_ascii=False),
        "retrieval_status": retrieval.get("status"),
        "retrieval_hits": len(retrieval.get("hits", [])),
        "context_status": context.get("status"),
        "context_groups": len(context.get("evidence_groups", [])),
        "context_characters": (context.get("budget") or {}).get("used"),
        "provider": generated.get("provider"),
        "returned_model": generated.get("returned_model"),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "generation_latency_ms": generated.get("latency_ms"),
        "pipeline_seconds": record.get("pipeline_seconds"),
    })

if overview:
    with (RUN_DIR / "results.csv").open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(overview[0]))
        writer.writeheader()
        writer.writerows(overview)
write_results(RUN_DIR, ordered_records)

status_counts = Counter(record["pipeline_status"] for record in ordered_records)
timings = [
    record["pipeline_seconds"] for record in ordered_records
    if isinstance(record.get("pipeline_seconds"), (int, float))
]
missing_keys = [key for key in ordered_keys if key not in records_by_key]
summary = {
    "run_schema": "bulk-answer-summary/v1",
    "run_dir": str(RUN_DIR.resolve()),
    "dataset": benchmark["dataset"],
    "benchmark_sha256": benchmark["sha256"],
    "available_questions": benchmark["question_count"],
    "selected_questions": len(selected_questions),
    "saved_records": len(ordered_records),
    "complete": not missing_keys,
    "missing_record_keys": missing_keys,
    "pipeline_status_counts": dict(sorted(status_counts.items())),
    "live": LIVE,
    "max_workers": MAX_WORKERS,
    "last_session_seconds": batch_seconds,
    "pipeline_seconds": {
        "median": statistics.median(timings) if timings else None,
        "p95": sorted(timings)[round((len(timings) - 1) * 0.95)] if timings else None,
        "samples": len(timings),
    },
    "updated_at_utc": datetime.now(UTC).isoformat(),
}
write_json(RUN_DIR / "summary.json", summary)
print("Save folder:", RUN_DIR.resolve())
print({
    "contexts": str(CONTEXTS_PATH.resolve()),
    "results_json": str((RUN_DIR / "results.json").resolve()),
    "results_csv": str((RUN_DIR / "results.csv").resolve()),
    "records": str(RECORDS_PATH.resolve()),
    "summary": str((RUN_DIR / "summary.json").resolve()),
})
summary

Save folder: C:\Users\PK\Desktop\projects\mobile_rag\artifacts\05_2_bulk_answer_generation\20260913_160742
{'contexts': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\05_2_bulk_answer_generation\\20260913_160742\\contexts.json', 'results_json': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\05_2_bulk_answer_generation\\20260913_160742\\results.json', 'results_csv': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\05_2_bulk_answer_generation\\20260913_160742\\results.csv', 'records': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\05_2_bulk_answer_generation\\20260913_160742\\records.jsonl', 'summary': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\05_2_bulk_answer_generation\\20260913_160742\\summary.json'}


{'run_schema': 'bulk-answer-summary/v1',
 'run_dir': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\05_2_bulk_answer_generation\\20260913_160742',
 'dataset': 'Q_S1',
 'benchmark_sha256': '8ca342ad54ecee63d357b6fb0034672dbd01fce73c0fbf84d2b8998e37207b93',
 'available_questions': 100,
 'selected_questions': 5,
 'saved_records': 5,
 'complete': True,
 'missing_record_keys': [],
 'pipeline_status_counts': {'api_error': 5},
 'live': True,
 'max_workers': 4,
 'last_session_seconds': 1.6848123000236228,
 'pipeline_seconds': {'median': 1.0096465999959037,
  'p95': 1.282591599971056,
  'samples': 5},
 'updated_at_utc': '2026-09-13T16:07:43.997876+00:00'}